#### Stage 1: Raw-data preprocessing


Define the helper functions used for raw-data preprocessing.


In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
import json
import re
import ast
from typing import Dict, List, Tuple, Union
from collections import defaultdict, Counter
from itertools import product

In [ ]:
def fix_restart_timestamps(df):
    # Copy the input and add an index column to preserve the original order
    df = df.copy()
    df['_original_index'] = range(len(df))
    
    result_dfs = []
    
    # Process each SEQID group while keeping the original order
    for _, group in df.groupby('SEQID', sort=False):
        # Sort back into the original order
        group = group.sort_values('_original_index').reset_index(drop=True)
        # Cast to float explicitly to avoid a FutureWarning
        adjusted_timestamps = group['timestamp'].astype(float).copy()
        
        # Locate the RESTART events
        restart_indices = group[group['event_type'] == 'RESTART'].index.tolist()
        
        for restart_idx in restart_indices:
            # Only handle RESTART when it is neither the first nor the last row
            if restart_idx > 0 and restart_idx < len(group) - 1:
                # Timestamp just before RESTART (already adjusted)
                prev_timestamp = adjusted_timestamps.iloc[restart_idx - 1]
                
                # Timestamp just after RESTART (original value)
                next_original_timestamp = group['timestamp'].iloc[restart_idx + 1]
                
                # RESTART timestamp = previous + (next original / 2)
                restart_timestamp = prev_timestamp + (next_original_timestamp / 2).round()
                adjusted_timestamps.iloc[restart_idx] = restart_timestamp
                
                # Find the END that follows RESTART
                end_idx = None
                for i in range(restart_idx + 1, len(group)):
                    if group['event_type'].iloc[i] == 'END':
                        end_idx = i
                        break
                
                # If no END is found, process until the end of the group
                if end_idx is None:
                    end_idx = len(group) - 1
                
                # Accumulate timestamps from after RESTART up to END
                # offset = adjusted RESTART timestamp
                offset = restart_timestamp
                
                for i in range(restart_idx + 1, end_idx + 1):
                    original_timestamp = group['timestamp'].iloc[i]
                    adjusted_timestamps.iloc[i] = offset + original_timestamp
        
        # Apply the adjusted timestamps
        group['timestamp'] = adjusted_timestamps
        result_dfs.append(group)
    
    # Concatenate all groups
    result = pd.concat(result_dfs, ignore_index=True)
    
    # Restore the original order and drop the temporary index column
    result = result.sort_values('_original_index').drop('_original_index', axis=1).reset_index(drop=True)
    
    return result

def count_sequences(df, time_window=100):
    """
    Count event combinations within a time window per SEQID.
    
    Parameters:
    df: DataFrame (SEQID, event_type, event_description, timestamp)
    time_window: time-window threshold (default: 100)
    
    Returns:
    Counter: count per combination
    """
    all_combinations = []
    
    for seqid, group_df in df.groupby('SEQID'):
        group_df = group_df.reset_index(drop=True)
        
        groups = []
        current_group = [0]
        
        for i in range(1, len(group_df)):
            if group_df.loc[i, 'timestamp'] - group_df.loc[current_group[0], 'timestamp'] <= time_window:
                current_group.append(i)
            else:
                groups.append(current_group)
                current_group = [i]
        groups.append(current_group)
        
        # Extract the combination for each group
        for group in groups:
            combo = tuple(group_df.loc[idx, 'event_type'] for idx in group)
            all_combinations.append(combo)
    
    return Counter(all_combinations)


def merge_patterns(df, pattern_map):
    """
    Merge a specific pattern into a single row.
    
    Parameters:
    df: DataFrame (SEQID, event_type, event_description, timestamp)
    pattern_map: dict {('A', 'B', 'C'): 'B', ...}  # value is the event_type to keep
    
    Returns:
    DataFrame: merged dataframe
    """
    result_rows = []
    
    for seqid, group_df in df.groupby('SEQID'):
        group_df = group_df.reset_index(drop=True)
        events = group_df['event_type'].tolist()
        
        i = 0
        while i < len(events):
            matched = False
            
            for pattern, keep_type in sorted(pattern_map.items(), key=lambda x: -len(x[0])):
                pattern_len = len(pattern)
                if i + pattern_len <= len(events):
                    if tuple(events[i:i+pattern_len]) == pattern:
                        # Find and keep the keep_type row within the pattern
                        for j in range(pattern_len):
                            if events[i+j] == keep_type:
                                result_rows.append(group_df.iloc[i+j])
                                break
                        i += pattern_len
                        matched = True
                        break
            
            if not matched:
                result_rows.append(group_df.iloc[i])
                i += 1
    
    return pd.DataFrame(result_rows).reset_index(drop=True)

def replace_pattern(df, pattern, new_event_type, new_event_description):
    """
    Replace a specific pattern with a new event_type / event_description.
    
    Parameters:
    df: DataFrame (SEQID, event_type, event_description, timestamp)
    pattern: tuple ('A', 'B', 'C')
    new_event_type: str 'NEW_TYPE'
    new_event_description: str 'NEW_DESC'
    
    Returns:
    DataFrame: transformed dataframe
    """
    result_rows = []
    pattern_len = len(pattern)
    
    for seqid, group_df in df.groupby('SEQID'):
        group_df = group_df.reset_index(drop=True)
        events = group_df['event_type'].tolist()
        
        i = 0
        while i < len(events):
            if i + pattern_len <= len(events):
                if tuple(events[i:i+pattern_len]) == pattern:
                    # Copy the first row and change only type/description
                    first_row = group_df.iloc[i].copy()
                    first_row['event_type'] = new_event_type
                    first_row['event_description'] = new_event_description
                    result_rows.append(first_row)
                    i += pattern_len
                    continue
            
            result_rows.append(group_df.iloc[i])
            i += 1
    
    return pd.DataFrame(result_rows).reset_index(drop=True)

def preprocess_keypress_data(df):
    """
    Merge consecutive KEYPRESS events into a single event.
    
    Parameters:
    output_file_path (str): output file path (.txt)
    """
    
    # Process each SEQID group
    processed_groups = []
    
    for seqid in df['SEQID'].unique():
        seqid_data = df[df['SEQID'] == seqid].copy()
        # Sort by timestamp
        seqid_data = seqid_data.sort_values('timestamp').reset_index(drop=True)
        
        processed_data = process_keypress_sequences(seqid_data)
        processed_groups.append(processed_data)
    
    # Concatenate all groups
    result_df = pd.concat(processed_groups, ignore_index=True)
    
    print(f"Processed data size: {len(result_df)} rows")
    print(f"Removed rows: {len(df) - len(result_df)}")
    
    return result_df

def process_keypress_sequences(df):
    """
    Handle consecutive KEYPRESS sequences within a single SEQID.
    
    Parameters:
    df (DataFrame): data for a single SEQID
    
    Returns:
    DataFrame: processed data
    """
    
    if len(df) == 0:
        return df
    
    result_rows = []
    i = 0
    
    while i < len(df):
        current_row = df.iloc[i].copy()
        
        # When the current row is a KEYPRESS
        if current_row['event_type'] == 'KEYPRESS':
            keypress_count = 1
            j = i + 1
            
            # Count consecutive KEYPRESS rows
            while j < len(df) and df.iloc[j]['event_type'] == 'KEYPRESS':
                keypress_count += 1
                j += 1
            
            # Set the first KEYPRESS row's event_description to count=N
            current_row['event_description'] = f'count={keypress_count}'
            result_rows.append(current_row)
            
            # Move the index past the consecutive KEYPRESS block
            i = j
            
        else:
            # Otherwise keep the row as-is
            result_rows.append(current_row)
            i += 1
    
    return pd.DataFrame(result_rows)

## Stage 0: Split the full log file into per-item files


In [ ]:
data = pd.read_csv('US_logdata.txt', sep='\t')

left = range(1, 3)   # 1 ~ 2
right = range(1, 8)  # 1 ~ 7
pairs = product(left, right)

for comb in pairs:
    i, j = comb
    
    mask = (data["booklet_id"] == f"PS{i}") & (data["item_id"] == j)
    prob_data = data.loc[mask].copy()
    
    US_code = prob_data["CNTRYID"].str.split("_", n=1).str[1].str[:2]
    prob_data["SEQID_unify"] = US_code + "_" + prob_data["SEQID"].astype("string")
    
    prob_data.to_csv(f"./input_data/us_ps{i}_{j}.txt", sep='\t', index=False)

## Convert action event_type / event_description


1. Run raw-data preprocessing for each item.
2. Inspect the processed actions and their per-action frequencies.


#### PS1_1

In [ ]:
# Per-item Stage 1 preprocessing
problem_num = 'ps1_1'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## Common submission actions ##
# Normalize the final-submission buttons: BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})

## ps1_1 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'ENVIRONMENT', 'MC_HELP_TOOLBAR', 'MAIL_SENT', 'MAIL_DELETED',
                                    'MC_HELP_MENUITEM', 'SORT_MENU', 'COPY', 'PASTE',
                                    'NEW_FOLDER', 'MC_SORT', 'DELETE_FOLDER'])].copy()

pattern_map = {
    ('FOLDER_VIEWED', 'MAIL_DROP', 'MAIL_MOVED'): 'MAIL_DROP',
    ('FOLDER_VIEWED', 'MAIL_MOVED', 'MAIL_DROP'): 'MAIL_DROP',
    ('BUTTON', 'MAIL_MOVED'): 'BUTTON',
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON',
    ('BREAKOFF', 'END') : 'BREAKOFF'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

In [ ]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count} times")

### PS1_2

In [ ]:
# Per-item Stage 1 preprocessing
problem_num = 'ps1_2'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## Common submission actions ##
# Normalize the final-submission buttons: BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})

## ps1_2 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'ENVIRONMENT','MC_HELP_TOOLBAR', 'MAIL_SENT', 'MAIL_DELETED',
                                    'MC_HELP_MENUITEM', 'SORT_MENU', 'COPY', 'PASTE', 
                                    'NEW_FOLDER', 'MC_SORT', 'DELETE_FOLDER', 'MAIL_COPIED'])].copy()

pattern_map = {
    ('FOLDER_VIEWED', 'MAIL_DROP', 'MAIL_MOVED'): 'MAIL_DROP',
    ('FOLDER_VIEWED', 'MAIL_MOVED', 'MAIL_DROP'): 'MAIL_DROP',
    ('BUTTON', 'MAIL_MOVED'): 'BUTTON',
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON',
    ('BREAKOFF', 'END') : 'BREAKOFF'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

In [ ]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count} times")

### PS1_3

In [ ]:
# Per-item Stage 1 preprocessing
problem_num = 'ps1_3'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## Common submission actions ##
# Normalize the final-submission buttons: BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})

## ps1_3 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'ENVIRONMENT', 'HISTORY_BACK', 'HISTORY_NEXT', 'SS_SORT','SS_SEARCH', 'BOOKMARK_ADD'])].copy()

pattern_map = {
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON',
    ('BREAKOFF', 'END') : 'BREAKOFF'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

In [ ]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count} times")

### PS1_4

In [ ]:
# Per-item Stage 1 preprocessing
problem_num = 'ps1_4'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## Common submission actions ##
# Normalize the final-submission buttons: BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})

## ps1_4 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'ENVIRONMENT'])].copy()


pattern_map = {
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON',
    ('BREAKOFF', 'END') : 'BREAKOFF'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

In [ ]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count} times")

### PS1_5

In [ ]:
# Per-item Stage 1 preprocessing
problem_num = 'ps1_5'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## Common submission actions ##
# Normalize the final-submission buttons: BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})

## ps1_5 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'ENVIRONMENT','HISTORY_NEXT','HISTORY_BACK','HISTORY_ADD',
                                    'COPY', 'BOOKMARK_ADD'])].copy()


pattern_map = {
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON',
    ('BREAKOFF', 'END') : 'BREAKOFF',
    ('COMBOBOX', 'INFORMATION', 'GLOBAL_VAR', 'TRANSLATION'): 'COMBOBOX'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

In [ ]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count} times")

### PS1_6

In [ ]:
# Per-item Stage 1 preprocessing
problem_num = 'ps1_6'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## Common submission actions ##
# Normalize the final-submission buttons: BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})


## Translation type, description
pattern = ('HISTORY_ADD', 'BOX_PRESS')
data = replace_pattern(data, pattern, 'TEXTLINK', 'id=u021_404')


## ps1_6 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'HISTORY_NEXT', 'HISTORY_BACK', 'HISTORY_ADD', 'BOX_PRESS'])].copy()


pattern_map = {
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON',
    ('BREAKOFF', 'END') : 'BREAKOFF'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

In [ ]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count} times")

### PS1_7

In [ ]:
# Per-item Stage 1 preprocessing
problem_num = 'ps1_7'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## Common submission actions ##
# Normalize the final-submission buttons: BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})


## ps1_7 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'ENVIRONMENT', 'SPLITSCREEN','MC_HELP_TOOLBAR','MC_HELP_MENUITEM',
                                    'COPY', 'PASTE'])].copy()


pattern_map = {
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON',
    ('BREAKOFF', 'END') : 'BREAKOFF'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

In [ ]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count} times")

### PS2_1

In [ ]:
# Per-item Stage 1 preprocessing
problem_num = 'ps2_1'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## Common submission actions ##
# Normalize the final-submission buttons: BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})


## ps2_1 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'ENVIRONMENT', 'COPY', 'PASTE', 'MC_HELP_MENUITEM',
                                    'SS_SEARCH', 'SS_SORT'])].copy()


pattern_map = {
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON',
    ('BREAKOFF', 'END') : 'BREAKOFF'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

In [ ]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count} times")

### PS2_2

In [ ]:
# Per-item Stage 1 preprocessing
problem_num = 'ps2_2'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## Common submission actions ##
# Normalize the final-submission buttons: BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})


## ps2_2 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'ENVIRONMENT', 'SS_SEARCH', 'SS_SORT'])].copy()


pattern_map = {
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

In [ ]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count} times")

### PS2_3

In [ ]:
# Per-item Stage 1 preprocessing
problem_num = 'ps2_3'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## Common submission actions ##
# Normalize the final-submission buttons: BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})


## ps2_3 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'ENVIRONMENT','SS_SEARCH', 'SS_SORT', 'HISTORY_ADD','HISTORY_BACK','HISTORY_NEXT',
                                    'BOOKMARK_ADD', 'BUYBOOK'])].copy()


pattern_map = {
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

In [ ]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count} times")

### PS2_4

In [ ]:
# Per-item Stage 1 preprocessing
problem_num = 'ps2_4'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## Common submission actions ##
# Normalize the final-submission buttons: BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})


## Translation type, description
patterns = [
    (('BUTTON', 'SUBMIT_RESERVATION_SUCCESS'), 'SUCCESS', 'id=submit'),
    (('BUTTON', 'SUBMIT_RESERVATION_FAILURE'), 'FAILURE', 'id=submit'),
    (('BUTTON', 'CHANGE_RESERVATION_SUCCESS'), 'SUCCESS', 'id=change'),
    (('BUTTON', 'CHANGE_RESERVATION_FAILURE'), 'FAILURE', 'id=change')
]

for pattern, new_type, new_desc in patterns:
    data = replace_pattern(data, pattern, new_type, new_desc)


## ps2_4 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'ENVIRONMENT', 'MC_HELP_TOOLBAR', 'BOOKMARK_TOOLBAR', 'COPY', 'HOME_TOOLBAR',
                                    'HISTORY_ADD', 'HISTORY_BACK', 'HISTORY_BACK_TOOLBAR', 'HISTORY_FORWARD_TOOLBAR', 'HISTORY_NEXT',
                                    'WP_HELP_MENUITEM', 'MC_HELP_MENUITEM', 'NEW_FOLDER', 'MC_SORT', 'BOOKMARK_ADD',
                                    'SORT_MENU', 'MAIL_COPIED', 'WB_HELP_TOOLBAR', 'WB_HELP_MENUITEM',
                                    'DELETE_FOLDER', 'COPY_MENUITEM', 'PASTE_MENUITEM', 'PASTE'])].copy()


pattern_map = {
    ('FOLDER_VIEWED', 'MAIL_DROP', 'MAIL_MOVED'): 'MAIL_DROP',
    ('FOLDER_VIEWED', 'MAIL_MOVED', 'MAIL_DROP'): 'MAIL_DROP',
    ('BUTTON', 'MAIL_MOVED'): 'BUTTON',
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON',
    ('BREAKOFF', 'END') : 'BREAKOFF'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

In [ ]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count} times")

### PS2_5

In [ ]:
# Per-item Stage 1 preprocessing
problem_num = 'ps2_5'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## Common submission actions ##
# Normalize the final-submission buttons: BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})


## ps2_5 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'MC_HELP_TOOLBAR', 'MAIL_SENT', 'PASTE', 'COPY', 'SORT_MENU',
                                    'MC_HELP_MENUITEM', 'MC_SORT', 'NEW_FOLDER', 'TRANSLATION', 'MAIL_DELETED'])].copy()


pattern_map = {
    ('FOLDER_VIEWED', 'MAIL_DROP', 'MAIL_MOVED'): 'MAIL_DROP',
    ('FOLDER_VIEWED', 'MAIL_MOVED', 'MAIL_DROP'): 'MAIL_DROP',
    ('BUTTON', 'MAIL_MOVED'): 'BUTTON',
    ('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON',
    ('BREAKOFF', 'END') : 'BREAKOFF'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

In [ ]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count} times")

### PS2_6

In [ ]:
# Per-item Stage 1 preprocessing
problem_num = 'ps2_6'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## Common submission actions ##
# Normalize the final-submission buttons: BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})


## ps2_6 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'SORT_MENU', 'MC_HELP_MENUITEM', 'PASTE', 'COPY', 'MC_SORT', 'DELETE_FOLDER',
                                    'MAIL_COPIED', 'MAIL_SENT', 'NEW_FOLDER', 'MAIL_DELETED', 'MC_HELP_TOOLBAR'])].copy()


pattern_map = {
('FOLDER_VIEWED', 'MAIL_DROP', 'MAIL_MOVED'): 'MAIL_DROP',
('FOLDER_VIEWED', 'MAIL_MOVED', 'MAIL_DROP'): 'MAIL_DROP',
('BUTTON', 'MAIL_MOVED'): 'BUTTON',
('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

In [ ]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count} times")

### PS2_7

In [ ]:
# Per-item Stage 1 preprocessing
problem_num = 'ps2_7'
file = 'us_' + problem_num + '.txt'
file_path = './input_data/' + file

path_1st = f'./input_data/1st_data/us_{problem_num}.pkl'

data = pd.read_csv(file_path, sep='\t')
data.drop(columns=['SEQID'],inplace=True)
data.rename(columns={'SEQID_unify' : 'SEQID'}, inplace=True)

data = fix_restart_timestamps(data)

## Common submission actions ##
# Normalize the final-submission buttons: BUTTON_next, BUTTON_ok, BUTTON_cancel
data['event_description'] = data['event_description'].replace({
    'id=nextInquiry_button' : 'id=next', #BUTTON_next
    'id=endtask_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt3' : 'id=ok', #BUTTON_ok
    'id=endunit_txt4' : 'id=cancel', #BUTTON_cancel
    'id=endtask_txt4' : 'id=cancel', #BUTTON_cancel
})

# Remap when a specific keyword is present
mask = data['event_description'].str.contains('u023_pg6_popup1', na=False)

data.loc[mask, 'event_description'] = (
    data.loc[mask, 'event_description']
        .str.replace(r'(?<=id=u023_)pg5', 'pg6', regex=True)
)

## Translation ps2_7
conditions = [
    ('HISTORY_ADD', 'pageid=unit23page3', 'TEXTLINK', 'id=u023_default_txt3'),
    ('HISTORY_ADD', 'pageid=unit23page4', 'TEXTLINK', 'id=u023_default_txt4'),
    ('HISTORY_ADD', 'pageid=unit23page5', 'TEXTLINK', 'id=u023_default_txt5'),
    ('HISTORY_ADD', 'pageid=unit23page6', 'TEXTLINK', 'id=u023_default_txt6'),
    ('HISTORY_ADD', 'pageid=unit23page7', 'TEXTLINK', 'id=u023_default_txt7'),
    ('HISTORY_ADD', 'pageid=unit23page8', 'TEXTLINK', 'id=u023_default_txt8')
]

for old_type, old_desc, new_type, new_desc in conditions:
    mask = (data['event_type'] == old_type) & (data['event_description'] == old_desc)
    data.loc[mask, 'event_type'] = new_type
    data.loc[mask, 'event_description'] = new_desc


## ps2_7 Reduced action
data = data[~data['event_type'].isin(['DOACTION','NEXT_INQUIRY','NEXT_BUTTON', 'CONFIRMATION_OPENED', 'CONFIRMATION_CLOSED',
                                    'ENVIRONMENT', 'MAIL_SENT', 'HISTORY_BACK', 'COPY', 'SORT_MENU', 'MC_SORT', 'HISTORY_ADD',
                                    'HISTORY_NEXT', 'MC_HELP_MENUITEM', 'PASTE', 'MC_HELP_TOOLBAR', 'BOOKMARK_ADD'])].copy()

pattern_map = {
('FOLDER_VIEWED', 'MAIL_DROP', 'MAIL_MOVED'): 'MAIL_DROP',
('FOLDER_VIEWED', 'MAIL_MOVED', 'MAIL_DROP'): 'MAIL_DROP',
('BUTTON', 'MAIL_MOVED'): 'BUTTON',
('BUTTON', 'NEXT_ITEM', 'END'): 'BUTTON',
('BREAKOFF', 'END') : 'BREAKOFF'
}

df_merged = merge_patterns(data, pattern_map)

df = df_merged.drop(columns=['CNTRYID','booklet_id', 'item_id','event_name'])

df.drop_duplicates(keep='first', inplace=True)
fin_data = preprocess_keypress_data(df)

fin_data.reset_index(drop=True, inplace=True)

fin_data.to_pickle(path_1st)

fin_data.head()

In [ ]:
combo_counts = count_sequences(df, time_window=100)

for combo, count in combo_counts.most_common():
    print(f"{' → '.join(combo)}: {count} times")

### Stage 2 & 3 preprocessing


In [ ]:
# US only
remove_vars_dict = {
    'ps1_1' : {'test_time', 'end', 'value'},
    'ps1_2' : {'test_time', 'end', 'value'},
    'ps1_3' : {'test_time', 'end', 'value'},
    'ps1_4' : {'test_time', 'end', 'value', 'href', 'target'},
    'ps1_5' : {'test_time', 'end', 'value', 'href'},
    'ps1_6' : {'test_time', 'end', 'value', 'href', 'target'},
    'ps1_7' : {'test_time', 'end', 'value'},
    'ps2_1' : {'test_time', 'end', 'value'},
    'ps2_2' : {'test_time', 'end', 'value'},
    'ps2_3' : {'test_time', 'end', 'value', 'href'},
    'ps2_4' : {'test_time', 'end', 'value', 'href', 'target'},
    'ps2_5' : {'test_time', 'end', 'value'},
    'ps2_6' : {'test_time', 'end', 'value'},
    'ps2_7' : {'test_time', 'end', 'value', 'href'}
}

def process_event_description(description):
    """
    Split event_description on special characters and filter the parts.
    
    Args:
        description (str): the original event_description
    
    Returns:
        list: filtered component list
    """
    
    # Split on special characters ('|', '*', ')
    components = re.split(r'[|*$]', str(description))
    
    # Drop empty strings
    components = [comp.strip() for comp in components if comp.strip()]
    
    # System variables to drop (extend as needed)
    remove_vars = remove_vars_dict[problem_num]
    
    # Additional detail values to drop
    remove_details = {'nan', ',', '.'}
    
    # Filtered component list
    filtered_components = []
    
    for comp in components:
        if '=' in comp or comp == 'end':
            # Extract the system variable before '='
            var_name = comp.split('=')[0].strip()
            
            # Keep if it is not in the drop list
            if var_name not in remove_vars:
                filtered_components.append(comp)
        else:
            # Also handle the case without '='
            if not any(k in comp for k in remove_details):
                filtered_components.append(comp)
    
    fin_string = '|'.join(filtered_components)
    
    return fin_string

## create_event_type_dict(df) (review pending)
def create_event_type_dict(df):
    """
    Build four dictionaries from event_type and event_desc_list.
    
    Parameters:
    df (pd.DataFrame): dataframe containing event_type and event_desc_list columns
    
    Returns:
    tuple: (unit_dict, unit_count_dict, token_dict, token_count_dict)
    """
    
    # Dictionaries that hold the results
    event_dict = defaultdict(set)  # unique units
    event_count_dict = defaultdict(lambda: defaultdict(int))  # unit counts
    token_dict = defaultdict(set)  # unique tokens
    token_count_dict = defaultdict(lambda: defaultdict(int))  # token counts
    
    # Iterate over the dataframe
    for _, row in df.iterrows():
        event_type = row['event_type']
        event_desc = row['event_desc_list']
        
        # Handle the unit (full event_desc)
        event_dict[event_type].add(event_desc)
        event_count_dict[event_type][event_desc] += 1
        
        # Handle tokens (parts split on |)
        tokens = event_desc.split('|')
        
        for token in tokens:
            token_dict[event_type].add(token)  # use add() instead of update()
            token_count_dict[event_type][token] += 1
    
    # Convert sets to lists
    unit_dict = {key: list(value) for key, value in event_dict.items()}
    token_unique_dict = {key: list(value) for key, value in token_dict.items()}
    
    # Convert defaultdicts to plain dicts
    unit_count_dict = {key: dict(value) for key, value in event_count_dict.items()}
    token_count_dict_final = {key: dict(value) for key, value in token_count_dict.items()}
    
    return unit_dict, unit_count_dict, token_unique_dict, token_count_dict_final

In [ ]:
## Load each item, build per-item unit/token dictionaries,
## merge them into a single combined dictionary, and save.
## -> produces the full unit/token combination dictionary

unit_dict_list = []
unit_cnt_list = []
token_dict_list = []
token_cnt_list = []

left = range(1, 3)   # 1 ~ 2
right = range(1, 8)  # 1 ~ 7
pairs = product(left, right)

# Initialize the dictionaries to be merged
merged_unit_dict = defaultdict(set)
merged_token_dict = defaultdict(set)
merged_token_cnt = defaultdict(lambda: defaultdict(int))
merged_unit_cnt = defaultdict(lambda: defaultdict(int))

for pair in pairs:
    i, j = pair
    
    problem_num = f'ps{i}_{j}'
    
    path_1st = f'input_data/1st_data/us_{problem_num}.pkl'
    path_2nd = f'input_data/2nd_data/us_{problem_num}.pkl'
    
    fin_data = pd.read_pickle(path_1st)
    
    fin_data.event_type = fin_data.event_type.str.lower()
    fin_data.event_description = fin_data.event_description.str.lower()
    
    fin_data['event_desc_list'] = fin_data['event_description'].apply(process_event_description).copy()
    
    unit_dict, unit_cnt, token_dict, token_cnt = create_event_type_dict(fin_data)

    unit_dict_list.append(unit_dict)
    unit_cnt_list.append(unit_cnt)
    token_dict_list.append(token_dict)
    token_cnt_list.append(token_cnt)
    
    fin_data.to_pickle(path_2nd)
    
    # ===== Save dictionaries per item =====
    token_unit_path = 'input_data/token_units/'
    
    # Per-item file names
    token_name_individual = f'us_token_{problem_num}.pkl'
    unit_name_individual = f'us_unit_{problem_num}.pkl'
    token_cnt_individual = f'us_token_cnt_{problem_num}.pkl'
    unit_cnt_individual = f'us_unit_cnt_{problem_num}.pkl'
    
    # Save per item
    with open(token_unit_path + token_name_individual, "wb") as f:
        pickle.dump(token_dict, f)
    
    with open(token_unit_path + unit_name_individual, "wb") as f:
        pickle.dump(unit_dict, f)
    
    with open(token_unit_path + token_cnt_individual, "wb") as f:
        pickle.dump(token_cnt, f)
    
    with open(token_unit_path + unit_cnt_individual, "wb") as f:
        pickle.dump(unit_cnt, f)
    # ======================================
    

# Merge unit_dict_list
for d in unit_dict_list:
    for category, unit_list in d.items():
        merged_unit_dict[category].update(unit_list)  # add all list elements to the set

# Merge token_dict_list
for d in token_dict_list:
    for category, token_list in d.items():
        merged_token_dict[category].update(token_list)  # add all list elements to the set

for d in token_cnt_list:
    for category, subdict in d.items():
        for key, count in subdict.items():
            merged_token_cnt[category][key] += count  # accumulate counts for the same id

for d in unit_cnt_list:
    for category, subdict in d.items():
        for key, count in subdict.items():
            merged_unit_cnt[category][key] += count  # accumulate counts for the same id


# Convert sets to lists
merged_token_dict = {cat: list(tokens) for cat, tokens in merged_token_dict.items()}
merged_unit_dict = {cat: list(units) for cat, units in merged_unit_dict.items()}
merged_token_cnt = {cat: dict(ids) for cat, ids in merged_token_cnt.items()}
merged_unit_cnt = {cat: dict(ids) for cat, ids in merged_unit_cnt.items()}

token_unit_path = './input_data/token_units/'

token_name = 'us_token_ps_N.pkl'
unit_name = 'us_unit_ps_N.pkl'
token_cnt_path = 'us_token_cnt_ps_N.pkl'
unit_cnt_path = 'us_unit_cnt_ps_N.pkl'

with open(token_unit_path + token_name, "wb") as f:
    pickle.dump(merged_token_dict, f)

with open(token_unit_path + unit_name, "wb") as f:
    pickle.dump(merged_unit_dict, f)
    
with open(token_unit_path + token_cnt_path, "wb") as f:
    pickle.dump(merged_token_cnt, f)

with open(token_unit_path + unit_cnt_path, "wb") as f:
    pickle.dump(merged_unit_cnt, f)


In [ ]:
# Action cleanup
maintain_list = ['mail_drag', 'mail_drop', 'folder_viewed', 'mail_viewed', 'menuitem',
                'get_help', 'restart', 'folder_unfolded', 'folder_folded', 'radio_btn',
                'breakoff', 'shortcut', 'checkbox', 'tab', 'success', 'failure']
LLM_list = ['toolbar', 'menu', 'button', 'textbox_onfocus', 'textbox_killfocus', 'combobox', 'textlink']

def parse_text_table(text_path: str) -> pd.DataFrame:
    """
    Parse the table in a text file into a dataframe.
    
    Args:
        text_path: path to the text file
    
    Returns:
        pd.DataFrame: dataframe with event_type, description, substitute, step2_output, fin_result columns
    """
    
    with open(text_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Extract the table lines
    table_lines = []
    for line in content.split('\n'):
        line = line.strip()
        if line and '|' in line:
            table_lines.append(line)
    
    if len(table_lines) < 2:
        raise ValueError("No text table found.")
    
    # Split header and data
    header_line = table_lines[0]
    data_lines = table_lines[2:]
    
    # Parse the header
    headers = [col.strip() for col in header_line.split('|') if col]
    
    # Parse the data
    data = []
    for line in data_lines:
        temp = re.sub(r'\\\|', '[[PIPE]]', line)
        
        parts = [p.strip() for p in temp.split('|') if p.strip()]
        row = [p.replace('[[PIPE]]', '|').replace('\\', '') for p in parts]

        if len(row) == len(headers):
            data.append(row)
        
    # Build the dataframe
    df = pd.DataFrame(data, columns=headers)
    
    # Convert the count column to numeric if present
    if 'count' in df.columns:
        df['count'] = pd.to_numeric(df['count'], errors='coerce')
    
    # Check required columns
    required_cols = ['event_type', 'description', 'substitute'] # column names adjusted
    missing_cols = [col for col in required_cols if col not in df.columns]

    print(f"Mapping table parsed - columns: {list(df.columns)}")
    
    return df

def transform_mapping_dict(mapping_df) -> pd.DataFrame:
    """
    Build a mapping dictionary from a mapping table for list-form event_description logs.
    
    Args:
        mapping_df: mapping table (requires event_type, description, result columns)

    Returns:
        pd.DataFrame: transformed dataframe
    """
    
    # Convert the mapping table to a dict for fast lookups
    mapping_dict = {}
    for _, row in mapping_df.iterrows():
        # (event_type, description) -> result mapping
        key = (row['event_type'], row['description'])
        mapping_dict[key] = row['substitute']
    
    print(f"Mapping dictionary built: {len(mapping_dict)} rules")
    
    return mapping_dict

def parse_description(desc):
    """Parse event_description into a dictionary."""
    result = {}
    if pd.isna(desc) or desc == '':
        return result
    
    pairs = desc.split('|')
    for pair in pairs:
        if '=' in pair:
            key, value = pair.split('=', 1)
            result[key] = value
    return result

def transform_event(row):
    event_type = row['event_type']
    event_desc = row['event_desc_list']
    
    # A. Convert '_' to '-' in event_type
    event_type_converted = event_type.replace('_', '-')
    
    # 1. Keep
    if event_type in maintain_list:
        parsed = parse_description(event_desc)
        
        detailed_info_list = []
        for value in parsed.values():
            converted_value = re.sub(r'(u\d+[A-Za-z0-9]*)(_|$)', r'\1-', value)
            detailed_info_list.append(converted_value)
            
        detailed_info = '_'.join(detailed_info_list)

        if detailed_info:
            return f"{event_type_converted}_{detailed_info}"
        else:
            return f"{event_type_converted}"
    
    # 2. Process    
    if event_type == 'keypress':
        parsed = parse_description(event_desc)
        count = int(parsed.get('count', 0))
        count_str = '10+' if count > 10 else str(count)
        return f"keypress{count_str}"
    
    if event_type == 'cell_change':
        parsed = parse_description(event_desc)
        detailed_info = list(parsed.values())[0] if parsed else ''
        # Convert content_spreadsheet_colad to content-spreadsheet-colad
        converted_info = detailed_info.replace('content_spreadsheet_colad', 'content-spreadsheet-colad')
        return f"cell-change_{converted_info}"
    
    # 3. LLM
    if event_type in LLM_list:
        key = (event_type, event_desc)
        if key in LLM_mapping:
            return f"{event_type_converted}_{LLM_mapping[key]}"
        else:
            # If no mapping exists, keep the original
            return f"{event_type_converted}_{event_desc}"
    
    # Other cases
    if event_desc:
        return f"{event_type_converted}_{event_desc}"
    else:
        return f"{event_type_converted}"


In [ ]:
from itertools import product

left = range(1, 3)   # 1 ~ 2
right = range(1, 8)  # 1 ~ 7
pairs = product(left, right)

for pair in pairs:
    i, j = pair
    
    print(pair)
    problem_num = f'ps{i}_{j}'
    
    # preprocessing paths
    path_2nd = f'input_data/2nd_data/us_{problem_num}.pkl'
    path_3rd = f'input_data/3rd_data/us_{problem_num}.pkl'

    # model-input paths
    IRT_path = f'model_input/IRT/test_us_{problem_num}.csv'
    HW2V_path = f'model_input/HW2V/test_us_{problem_num}.txt'
    
    fin_data = pd.read_pickle(path_2nd)
    
    mapping_text_path = f'LLM/LLM_results/{problem_num}/step1_new_table.txt'
    mapping_df = parse_text_table(mapping_text_path)
    
    LLM_mapping = transform_mapping_dict(mapping_df)

    fin_data['processed_event'] = fin_data.apply(transform_event, axis=1)
    
    ## - preprocessing adjustment - ##
    # Remove rows whose processed_event is 'start'
    remove_list = ['start', 'button_next', 'button_ok', 'end']
    fin_data = fin_data[~fin_data['processed_event'].isin(remove_list)]
    ##############################################################
    
    # Use a lambda
    fin_data['processed_event'] = fin_data['processed_event'].apply(
        lambda x: '_'.join(dict.fromkeys(x.split('_')))
    )
    
    result_df = fin_data[['SEQID', 'event_type', 'event_desc_list', 'timestamp', 'processed_event']]
    
    result_df.to_pickle(path_3rd)
    
    sequence_data = []

    for seqid, group in result_df.groupby('SEQID'):
        # Join timestamp-ordered processed_event values with spaces
        events = group['processed_event'].tolist()
        seq_event = ' '.join(events)
        event_count = len(events)
        
        sequence_data.append({
            'SEQID': seqid,
            'seq_event': seq_event,
            'event_count': event_count
        })

    sequence_df = pd.DataFrame(sequence_data)
    
    sequence_df.to_csv(IRT_path, index=False)
    
    # Keep only non-empty rows from the action sequence column
    action_sequences = sequence_df['seq_event'].dropna()

    # Save to a text file (one sequence per line)
    with open(HW2V_path, 'w', encoding='utf-8') as f:
        for sequence in action_sequences:
            f.write(str(sequence) + '\n')

In [ ]:
import pandas as pd
from itertools import product

left = range(1, 3)   # 1 ~ 2
right = range(1, 8)  # 1 ~ 7
pairs = product(left, right)

summary_list = []

for pair in pairs:
    i, j = pair
    
    print(pair)
    problem_num = f'ps{i}_{j}'
    
    data = pd.read_csv(f"model_input/IRT/test_us_{problem_num}.csv")

    desc = data['event_count'].describe()
    desc['name'] = problem_num
    
    summary_list.append(desc)

combined_summary_list = pd.concat(summary_list, axis=1)
combined_summary_list.to_csv('paper_stats/action_seq.csv')
